# 03 · Feature Engineering

Adds engineered features, encodes categorical columns, and saves model-ready feature (X), target (y) and ID files.

## 1. Load cleaned data

In [3]:
import pandas as pd

path = '../data/'

train = pd.read_csv(path + 'borrower_clean_train.csv')
val = pd.read_csv(path + 'borrower_clean_val.csv')
test = pd.read_csv(path + 'borrower_clean_test.csv')

## 2. Engineered features

- `EXT_SOURCE_MEAN`: average of the three external scores
- `HIGH_DTI_FLAG`: 1 if the loan-to-income ratio is above the training 90th percentile

In [5]:
for data in (train, val, test):
    data['EXT_SOURCE_MEAN'] = (
        data['EXT_SOURCE_1'] + data['EXT_SOURCE_2'] + data['EXT_SOURCE_3']
    ) / 3

dti_threshold = train['debt_to_income'].quantile(0.90)
print("DTI 90th percentile threshold (from train):", round(dti_threshold, 3))

for data in (train, val, test):
    data['HIGH_DTI_FLAG'] = (data['debt_to_income'] > dti_threshold).astype(int)

print("Borrowers flagged HIGH_DTI - train:", train['HIGH_DTI_FLAG'].sum(),
      " val:", val['HIGH_DTI_FLAG'].sum(),
      " test:", test['HIGH_DTI_FLAG'].sum())

DTI 90th percentile threshold (from train): 7.486
Borrowers flagged HIGH_DTI - train: 21522  val: 4646  test: 4623


## 3. Binary encoding

In [7]:
for data in (train, val, test):
    data['gender'] = data['gender'].map({'M': 1, 'F': 0, 'XNA': 0})
    data['owns_car'] = data['owns_car'].map({'Y': 1, 'N': 0})
    data['owns_property'] = data['owns_property'].map({'Y': 1, 'N': 0})

## 4. One-hot encoding

The category list for each column is fixed from the training set, so validation and test always end up with exactly the same columns.

In [9]:
cat_cols = ['employment_type', 'education', 'family_status']
train_categories = {col: sorted(train[col].unique().tolist()) for col in cat_cols}

for data in (train, val, test):
    for col in cat_cols:
        data[col] = pd.Categorical(data[col], categories=train_categories[col])

train = pd.get_dummies(train, columns=cat_cols, drop_first=True, dtype=int)
val = pd.get_dummies(val, columns=cat_cols, drop_first=True, dtype=int)
test = pd.get_dummies(test, columns=cat_cols, drop_first=True, dtype=int)

print("Shape after encoding - train:", train.shape, " val:", val.shape, " test:", test.shape)
print("Columns match across all three:",
      list(train.columns) == list(val.columns) == list(test.columns))

Shape after encoding - train: (215257, 39)  val: (46127, 39)  test: (46127, 39)
Columns match across all three: True


## 5. Save engineered files

In [11]:
train.to_csv(path + 'borrower_engineered_train.csv', index=False)
val.to_csv(path + 'borrower_engineered_val.csv', index=False)
test.to_csv(path + 'borrower_engineered_test.csv', index=False)

## 6. Split into features / target / ID

In [13]:
def split_x_y_id(data):
    ids = data['SK_ID_CURR']
    y = data['TARGET']
    X = data.drop(columns=['SK_ID_CURR', 'TARGET'])
    return X, y, ids

X_train, y_train, ids_train = split_x_y_id(train)
X_val, y_val, ids_val = split_x_y_id(val)
X_test, y_test, ids_test = split_x_y_id(test)

print("Feature matrix shape - train:", X_train.shape,
      " val:", X_val.shape, " test:", X_test.shape)
print("Default rate - train:", round(y_train.mean() * 100, 2),
      " val:", round(y_val.mean() * 100, 2),
      " test:", round(y_test.mean() * 100, 2))

Feature matrix shape - train: (215257, 37)  val: (46127, 37)  test: (46127, 37)
Default rate - train: 8.07  val: 8.07  test: 8.07


## 7. Save model-ready files

In [15]:
X_train.to_csv(path + 'X_train.csv', index=False)
X_val.to_csv(path + 'X_val.csv', index=False)
X_test.to_csv(path + 'X_test.csv', index=False)

y_train.to_csv(path + 'y_train.csv', index=False)
y_val.to_csv(path + 'y_val.csv', index=False)
y_test.to_csv(path + 'y_test.csv', index=False)

ids_train.to_csv(path + 'ids_train.csv', index=False)
ids_val.to_csv(path + 'ids_val.csv', index=False)
ids_test.to_csv(path + 'ids_test.csv', index=False)

print("Saved X / y / ID files to", path)

Saved X / y / ID files to ../data/
